# Lab 3.3 - Custom Arty overlay and driver
Start a PYNQ Linux SD image, then copy this `notebooks/` directory beside `exported/`. Loading `lab3.bit` replaces the PL configuration. The separate bare-metal BOOT.BIN does not run Jupyter.

In [ ]:
from pynq import Overlay, DefaultIP
from pathlib import Path
bit = Path("../exported/lab3.bit")
assert bit.exists() and bit.with_suffix(".hwh").exists()
overlay = Overlay(str(bit))
print(overlay.ip_dict)
assert overlay.is_loaded()

In [ ]:
add_ip = overlay.scalar_add
print(add_ip.register_map)
add_ip.register_map.a = 3
add_ip.register_map.b = 4
assert int(add_ip.register_map.c) == 7
print("3 + 4 =", int(add_ip.register_map.c))
add_ip.write(0x10, 4)
add_ip.write(0x18, 5)
assert add_ip.read(0x20) == 9
print("4 + 5 =", add_ip.read(0x20))

In [ ]:
class AddDriver(DefaultIP):
    bindto = ["xilinx.com:hls:add:1.0"]
    def __init__(self, description):
        super().__init__(description=description)
    def add(self, a, b):
        self.write(0x10, a & 0xffffffff)
        self.write(0x18, b & 0xffffffff)
        return self.read(0x20)

overlay = Overlay(str(bit))
result = overlay.scalar_add.add(15, 20)
assert result == 35
print("Driver: 15 + 20 =", result)